# Extract, clean, and chunk 100 MyScheme PDFs

The notebook downloads 100 PDFs from `shrijayan/gov_myscheme`, removes common PDF-extraction artifacts without discarding scheme content, and creates retrieval-ready overlapping chunks.

In [4]:
%pip install -q huggingface_hub pypdf pandas tqdm

from collections import Counter
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
from huggingface_hub import HfApi, snapshot_download
from pypdf import PdfReader
from tqdm.auto import tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
DATASET_ID = "shrijayan/gov_myscheme"
MAX_PDFS = 100
DOWNLOAD_DIR = Path("data/gov_myscheme_pdfs")
OUTPUT_DIR = Path("data/gov_myscheme_extracted")

api = HfApi()
pdf_files = sorted(
    file for file in api.list_repo_files(DATASET_ID, repo_type="dataset")
    if file.lower().endswith(".pdf")
)[:MAX_PDFS]

if len(pdf_files) < MAX_PDFS:
    raise RuntimeError(f"Only found {len(pdf_files)} PDFs in {DATASET_ID}.")

snapshot_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    allow_patterns=pdf_files,
    local_dir=DOWNLOAD_DIR,
    max_workers=4,
)

pdf_paths = [DOWNLOAD_DIR / file for file in pdf_files]
print(f"Downloaded {len(pdf_paths)} PDFs to: {DOWNLOAD_DIR.resolve()}")


Fetching 100 files: 100%|██████████| 100/100 [00:00<00:00, 476.61it/s]

Downloaded 100 PDFs to: D:\Projects\Scheme Bot\data\gov_myscheme_pdfs


In [6]:
def extract_pages(pdf_path: Path) -> list[str]:
    reader = PdfReader(str(pdf_path))
    return [page.extract_text() or "" for page in reader.pages]

def clean_pdf_pages(pages: list[str]) -> str:
    """Clean PDF artifacts while retaining paragraph and list boundaries."""
    normalized_pages = []
    for page in pages:
        page = unicodedata.normalize("NFKC", page).replace("\x00", "")
        page = page.replace("\r\n", "\n").replace("\r", "\n")
        # Join words split at a line ending, e.g. "eligibi-\nlity".
        page = re.sub(r"(?<=[A-Za-z])-[ \t]*\n[ \t]*(?=[a-z])", "", page)
        lines = [re.sub(r"[ \t]+", " ", line).strip() for line in page.split("\n")]
        normalized_pages.append([line for line in lines if line])

    # Exact lines repeated on at least 30% of pages are normally headers/footers.
    line_pages = Counter({
        line: sum(line in set(page) for page in normalized_pages)
        for line in {line for page in normalized_pages for line in page}
    })
    minimum_repetitions = max(3, (len(normalized_pages) * 3 + 9) // 10)
    repeated_lines = {line for line, count in line_pages.items() if count >= minimum_repetitions}

    cleaned_pages = []
    for page in normalized_pages:
        lines = [
            line for line in page
            if line not in repeated_lines and not re.fullmatch(r"(?:page )?\d+(?: of \d+)?", line, flags=re.I)
        ]
        cleaned_pages.append("\n".join(lines))

    text = "\n\n".join(page for page in cleaned_pages if page)
    # Preserve paragraphs, but turn extraction-only newlines within a paragraph into spaces.
    paragraphs = [re.sub(r"\s*\n\s*", " ", paragraph).strip() for paragraph in re.split(r"\n{2,}", text)]
    return "\n\n".join(paragraph for paragraph in paragraphs if paragraph)

def chunk_text(text: str, chunk_words: int = 350, overlap_words: int = 50) -> list[str]:
    """Create ~350-word chunks with a 50-word overlap for retrieval."""
    words = text.split()
    step = chunk_words - overlap_words
    return [
        " ".join(words[start:start + chunk_words])
        for start in range(0, len(words), step)
        if words[start:start + chunk_words]
    ]


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
documents, chunks = [], []

for pdf_path in tqdm(pdf_paths, desc="Extracting and cleaning PDFs"):
    source_file = str(pdf_path.relative_to(DOWNLOAD_DIR))
    document = {"source_file": source_file}
    try:
        pages = extract_pages(pdf_path)
        raw_text = "\n".join(pages)
        clean_text = clean_pdf_pages(pages)
        document.update({
            "page_count": len(pages),
            "raw_character_count": len(raw_text),
            "character_count": len(clean_text),
            "text": clean_text,
            "error": None,
        })
        chunks.extend({
            "source_file": source_file,
            "chunk_id": chunk_id,
            "text": chunk,
        } for chunk_id, chunk in enumerate(chunk_text(clean_text)))
    except Exception as exc:
        document.update({
            "page_count": 0, "raw_character_count": 0, "character_count": 0,
            "text": "", "error": f"{type(exc).__name__}: {exc}",
        })
    documents.append(document)

document_file = OUTPUT_DIR / "myscheme_100_clean_documents.jsonl"
chunk_file = OUTPUT_DIR / "myscheme_100_chunks.jsonl"
for path, rows in ((document_file, documents), (chunk_file, chunks)):
    with path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = pd.DataFrame(documents).drop(columns="text")
summary_file = OUTPUT_DIR / "myscheme_100_pdfs_summary.csv"
summary.to_csv(summary_file, index=False)

print(f"Cleaned {(summary.error.isna()).sum()} of {len(summary)} PDFs into {len(chunks):,} chunks.")
print(f"Clean documents: {document_file.resolve()}")
print(f"Chunks: {chunk_file.resolve()}")
print(f"Summary: {summary_file.resolve()}")
display(summary)


Extracting and cleaning PDFs: 100%|██████████| 100/100 [00:26<00:00,  3.83it/s]

Cleaned 100 of 100 PDFs into 474 chunks.
Clean documents: D:\Projects\Scheme Bot\data\gov_myscheme_extracted\myscheme_100_clean_documents.jsonl
Chunks: D:\Projects\Scheme Bot\data\gov_myscheme_extracted\myscheme_100_chunks.jsonl
Summary: D:\Projects\Scheme Bot\data\gov_myscheme_extracted\myscheme_100_pdfs_summary.csv


,source_file,page_count,raw_character_count,character_count,error
0,text_data\25-ciss copy.pdf,2,12507,12504,None
1,text_data\25-ciss.pdf,2,12507,12504,None
2,text_data\40shydcs copy.pdf,2,9659,9660,None
3,text_data\40shydcs.pdf,2,9659,9660,None
4,text_data\a-gainer copy.pdf,2,12825,12825,None
...,...,...,...,...,...
95,text_data\agr3fmsstf copy.pdf,2,10543,10541,None
96,text_data\agr3fmsstf.pdf,2,10543,10541,None
97,text_data\agr4fmsscf copy.pdf,2,9444,9445,None
98,text_data\agr4fmsscf.pdf,2,9444,9445,None
